In [172]:
import statsmodels.api as sm
import numpy as np
import pandas as pd
from sklearn.preprocessing import  MinMaxScaler
from sklearn.preprocessing import  OneHotEncoder 

# importar el data set
dataset = pd.read_csv('50_Startups.csv')
X = dataset.iloc[:, :-1]
y = dataset.iloc[:, 4].values
sl = 0.05
features = list(dataset.columns.values)
features.remove('Profit')
print(features)
dataset.head(5)

['R&D Spend', 'Administration', 'Marketing Spend', 'State']


,R&D Spend,Administration,Marketing Spend,State,Profit
0,165349.20,136897.80,471784.10,New York,192261.83
1,162597.70,151377.59,443898.53,California,191792.06
2,153441.51,101145.55,407934.54,Florida,191050.39
3,144372.41,118671.85,383199.62,New York,182901.99
4,142107.34,91391.77,366168.42,Florida,166187.94


In [173]:
# determine categorical and numerical features
numerical = X.select_dtypes(include=['int64', 'float64'])
categorical = X.select_dtypes(include=['object', 'bool'])
[features.remove(name) for name in categorical.columns.tolist()]
enc = OneHotEncoder(categories='auto')
onehotlabels = enc.fit_transform(categorical).toarray()
onehotlabels = onehotlabels[:,1:]  # Evitando co-liealidad

# Normalizar solo datos numericos
X = numerical.iloc[:,:].values
sc_X = MinMaxScaler()
x_norm = sc_X.fit_transform(X)

##### Concatenar onehotlabels con X 
X_completo = np.concatenate((onehotlabels,x_norm),axis=1)
X_df = pd.DataFrame(X_completo)
X_df.columns = [f'FeatureHot{i}' for i, _ in enumerate(range(onehotlabels.shape[1]))] + features  

In [174]:
X_df.head(5)

,FeatureHot0,FeatureHot1,R&D Spend,Administration,Marketing Spend
0,0.0,1.0,1.000000,0.651744,1.000000
1,0.0,0.0,0.983359,0.761972,0.940893
2,1.0,0.0,0.927985,0.379579,0.864664
3,0.0,1.0,0.873136,0.512998,0.812235
4,1.0,0.0,0.859438,0.305328,0.776136


## **Tarea**

1. Aplicar statsmodels Ordinary Least Square (OLS) a dataframe
2. Analizar p-value de cada variables independientes
3. Aplicar algoritmo de eliminación hacia atrás
4. Analizar resultados desde el punto de vista de
               R-Cuadrado y R-Cuadrado ajustado
     

## 1. Aplicar statsmodels Ordinary Least Square (OLS) a dataframe

## 2. Analizar p-value de cada variables independientes

In [175]:
regressor_OLS = sm.OLS(y, X_df).fit()    
summary_tables = regressor_OLS.summary().tables
coef_table = summary_tables[1]
print(f"\nMÉTRICAS: R² = {regressor_OLS.rsquared:.4f}, R²-adj = {regressor_OLS.rsquared_adj:.4f}")
print(coef_table.as_text())   


MÉTRICAS: R² = 0.9807, R²-adj = 0.9786
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
FeatureHot0      7787.0351   6072.700      1.282      0.206   -4444.010       2e+04
FeatureHot1      1.052e+04   5709.861      1.842      0.072    -981.188     2.2e+04
R&D Spend        1.203e+05    1.4e+04      8.613      0.000    9.22e+04    1.48e+05
Administration    4.94e+04   8533.093      5.789      0.000    3.22e+04    6.66e+04
Marketing Spend  4.911e+04   1.35e+04      3.647      0.001     2.2e+04    7.62e+04


In [176]:
def backwardElimination(y, X, sl):
    X_temp = X.copy()
    while True:
        regressor_OLS = sm.OLS(y, X_temp).fit()    
        maxVar = max(regressor_OLS.pvalues)
        maxVar_name = regressor_OLS.pvalues.idxmax()
        print(f"MÉTRICAS: R² = {regressor_OLS.rsquared:.4f}, R²-adj = {regressor_OLS.rsquared_adj:.4f}")
        print(regressor_OLS.summary().tables[1])
        print()
        if maxVar > sl:
            X_temp = X_temp.drop(maxVar_name, axis=1)
        else:
            break        

In [177]:
backwardElimination(y,X_df, sl)

MÉTRICAS: R² = 0.9807, R²-adj = 0.9786
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
FeatureHot0      7787.0351   6072.700      1.282      0.206   -4444.010       2e+04
FeatureHot1      1.052e+04   5709.861      1.842      0.072    -981.188     2.2e+04
R&D Spend        1.203e+05    1.4e+04      8.613      0.000    9.22e+04    1.48e+05
Administration    4.94e+04   8533.093      5.789      0.000    3.22e+04    6.66e+04
Marketing Spend  4.911e+04   1.35e+04      3.647      0.001     2.2e+04    7.62e+04

MÉTRICAS: R² = 0.9800, R²-adj = 0.9783
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
FeatureHot1      7159.8718   5108.859      1.401      0.168   -3123.728    1.74e+04
R&D Spend        1.191e+05    1.4e+04      8.485      0.000    9.08e+04    1.47e+